In [7]:
from models.groq_openai import GetGroqModelClient
from models.openrouter import getOpenRouterModel
from dotenv import load_dotenv
load_dotenv()
llm=GetGroqModelClient()
model=getOpenRouterModel()

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional
from dotenv import load_dotenv
load_dotenv()

class JDDetails(BaseModel):
    job_title: str = Field(description="Job title")
    preferred_skills: List[str] = Field(description="Preferred skills. Use 'Not Found' if missing.")
    required_skills: List[str] = Field(description="Required skills. Use 'Not Found' if missing.")
    experience_years: int = Field(description="Work experience. Use 'Not Found' if missing.")
    education: str = Field(description="Education details. Use 'Not Found' if missing.")
    keywords: List[str] = Field(description="Important keywords extracted from resume")


def process_jd(llm,jd: str) -> dict:
    """Extract structured data from job description"""
    jd_text = jd
    output_parser = JsonOutputParser(pydantic_object=JDDetails)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract job description details..."),
        ("human", "Job description text:\n\n{jd_text}\n\n{format_instructions}")
    ])
    
    chain = prompt | llm | output_parser
    
    return chain.invoke({
        "jd_text": jd_text,
        "format_instructions": output_parser.get_format_instructions()
    })

In [9]:
jd="""Job Title: Junior Machine Learning Engineer

Location: Bangalore / Hybrid
Experience Required: 1–3 Years

Role Overview

We are looking for a Junior Machine Learning Engineer to build and deploy scalable ML systems for real-world applications. The role involves working on data pipelines, model optimization, and integrating machine learning solutions into production environments.

Key Responsibilities
Develop and deploy machine learning models using Python
Work on NLP-based tasks such as text classification and entity recognition
Design data pipelines for preprocessing and feature engineering
Optimize model performance and scalability
Collaborate with backend teams to integrate ML models via APIs
Monitor model performance in production
Required Skills
Strong proficiency in Python
Good understanding of machine learning fundamentals
Experience with scikit-learn and PyTorch
Knowledge of NLP concepts and transformer models
Experience working with APIs (FastAPI / Flask)
Familiarity with Git and version control
Preferred Qualifications
Experience with SQL and relational databases
Familiarity with Docker and containerization
Exposure to cloud platforms (AWS / GCP / Azure)
Understanding of CI/CD pipelines for ML (MLOps)
Experience with data engineering tools (Airflow, Spark)
Education
B.Tech / B.E. in Computer Science, Data Science, or related field
Additional Notes
Candidates with strong project experience in ML/NLP are encouraged to apply
Hands-on experience is preferred over theoretical knowledge"""

resume_path=r"C:\Users\hardi\OneDrive\Desktop\ats_resume_scorer\hardik_resume.pdf"

In [10]:
jd_details=process_jd(llm,jd)

In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional
from dotenv import load_dotenv
load_dotenv()

class ExperienceEntry(BaseModel):
    title: str = Field(description="Job title or role")
    description: str = Field(description="Work done or responsibilities")
    duration: str = Field(description="Duration (e.g., 2025-2026)")

class ProjectEntry(BaseModel):
    name: str = Field(description="Project name")
    tech_stack: List[str] = Field(description="Technologies used in the project")
    description: str = Field(description="What the project does")

class EducationEntry(BaseModel):
    degree: str = Field(description="Degree name (e.g., B.Tech)")
    field: str = Field(description="Field of study (e.g., Computer Science)")
    institution: str = Field(description="College or school name")
    duration: str = Field(description="Years attended")

class Skills(BaseModel):
    technical: List[str] = Field(description="Programming languages and frameworks")
    tools: List[str] = Field(description="Tools and platforms (Docker, AWS, Git, etc.)")
    concepts: List[str] = Field(description="Concepts (ML, NLP, RAG, etc.)")

class ResumeDetails(BaseModel):
    name: str = Field(description="Full name of the candidate")
    email: str = Field(description="Email address")
    phone: str = Field(description="Phone number")
    skills: Skills
    experience: List[ExperienceEntry] = Field(description="List of work or project experiences")
    total_experience_years: Optional[float] = Field(description="Total years of experience if available")
    projects: List[ProjectEntry]
    education: List[EducationEntry]
    certifications: List[str] = Field(description="List of certifications. Empty list if none.")
    keywords: List[str] = Field(description="Important keywords extracted from resume")

def load_resume(file_path: str):
    if file_path.endswith(".pdf"):
        return PyPDFLoader(file_path)
    elif file_path.endswith(".docx"):
        return Docx2txtLoader(file_path)
    elif file_path.endswith(".txt"):
        return TextLoader(file_path)
    else:
        raise ValueError("Unsupported resume format")

def process_resume(llm,resume_path: str) -> dict:
    """Extract structured data from resume"""
    
    loader = load_resume(resume_path)
    docs = loader.load()
    
    full_text = "\n\n".join([doc.page_content for doc in docs])
    
    output_parser = JsonOutputParser(pydantic_object=ResumeDetails)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract resume details..."),
        ("human", "Resume text:\n\n{resume_text}\n\n{format_instructions}")
    ])
    
    chain = prompt | llm | output_parser
    
    return chain.invoke({
        "resume_text": full_text,
        "format_instructions": output_parser.get_format_instructions()
    })

In [12]:
resume_details = process_resume(llm, resume_path)

In [13]:
jd_details

{'job_title': 'Junior Machine Learning Engineer',
 'preferred_skills': ['SQL and relational databases',
  'Docker and containerization',
  'Cloud platforms (AWS, GCP, Azure)',
  'CI/CD pipelines for ML (MLOps)',
  'Data engineering tools (Airflow, Spark)'],
 'required_skills': ['Python',
  'Machine learning fundamentals',
  'scikit-learn',
  'PyTorch',
  'NLP concepts and transformer models',
  'APIs (FastAPI / Flask)',
  'Git and version control'],
 'experience_years': 1,
 'education': 'B.Tech / B.E. in Computer Science, Data Science, or related field',
 'keywords': ['Machine Learning',
  'Python',
  'NLP',
  'text classification',
  'entity recognition',
  'data pipelines',
  'feature engineering',
  'model optimization',
  'scalable ML systems',
  'APIs',
  'FastAPI',
  'Flask',
  'Git',
  'scikit-learn',
  'PyTorch',
  'transformer models',
  'SQL',
  'Docker',
  'AWS',
  'GCP',
  'Azure',
  'CI/CD',
  'MLOps',
  'Airflow',
  'Spark',
  'Bangalore',
  'Hybrid']}

In [14]:
from autogen_agentchat.agents import AssistantAgent
from models.openrouter import getOpenRouterModel

SYSTEM_MESSAGE = """
You are an expert ATS (Applicant Tracking System) scorer.

CRITICAL INSTRUCTIONS FOR SKILL MATCHING:
1. Match skills SEMANTICALLY, not just exact text:
   - "Python" matches "Python programming", "Proficiency in Python", "Python development"
   - "ML" matches "Machine Learning", "machine learning fundamentals"
   - "TensorFlow or PyTorch" matches if candidate has EITHER one
   - "NLP" matches "Natural Language Processing", "NLP concepts"
   - "Docker" matches "Docker containers", "Docker deployment", "Familiarity with Docker"

2. SCORING BREAKDOWN (Total: 100 points):
   - Required Skills Match (40 points): Count how many required skills the candidate has
   - Preferred Skills Match (20 points): Count how many preferred/nice-to-have skills
   - Experience Relevance (20 points): Do projects show relevant experience?
   - Education Match (10 points): Degree in relevant field?
   - Additional Factors (10 points): Certifications, achievements, etc.

3. MATCHING RULES:
   - If candidate has the skill under ANY name/variation, count it as MATCHED
   - If candidate has a BETTER version (e.g., has PyTorch when TensorFlow is required), count it
   - If candidate has similar/related skill (e.g., has FastAPI when Flask is required), give 0.5 credit
   - Projects using a skill COUNT as having that skill

4. OUTPUT FORMAT (JSON):
{
  "total_score": <0-100>,
  "classification": "<Excellent Match (80+) | Good Match (60-79) | Fair Match (40-59) | Poor Match (<40)>",
  "breakdown": {
    "required_skills": <0-40>,
    "preferred_skills": <0-20>,
    "experience": <0-20>,
    "education": <0-10>,
    "additional": <0-10>
  },
  "matched_required": <count>,
  "total_required": <count>,
  "matched_preferred": <count>,
  "total_preferred": <count>,
  "matched_skills_detail": [
    {"skill": "Python", "found_as": "Python programming", "in_section": "skills"},
    {"skill": "Machine Learning", "found_as": "ML", "in_section": "projects"}
  ],
  "missing_skills": ["skill1", "skill2"],
  "recommendations": "Specific suggestions for improvement"
}

EXAMPLE:
JD requires: "Python, TensorFlow or PyTorch, NLP, Docker"
Candidate has: "Python programming, PyTorch framework, Natural Language Processing, Docker containers"
Result: 4/4 matched (100% match on required skills)

BE GENEROUS in matching - if the skill is there in ANY form, count it!
"""

def getATS_scorerAgent(model):
    ats_scorer_agent = AssistantAgent(
        name="ATS_Scorer",
        model_client=model,
        description="An agent that scores resumes against job descriptions.",
        system_message=SYSTEM_MESSAGE)
    return ats_scorer_agent

In [15]:
import json

task = f"""
ResumeDetails:{resume_details}

JDDetails:{jd_details}
"""
ats_agent = getATS_scorerAgent(model)
score = await ats_agent.run(task=task)

In [16]:
response=score.messages[1].content

In [17]:
import json
import re
from typing import Dict, Any, Optional

def parse_ats_response(response_text: str) -> Optional[Dict[str, Any]]:
    """
    Parse ATS scorer JSON response
    
    Handles both:
    - Pure JSON
    - JSON wrapped in markdown code blocks
    - JSON with text before/after
    """
    
    try:
        # Try direct JSON parse first
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass
    
    # Try to extract JSON from markdown code blocks
    json_pattern = r'```(?:json)?\s*(\{.*?\})\s*```'
    match = re.search(json_pattern, response_text, re.DOTALL)
    
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    
    # Try to find raw JSON in text
    json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    match = re.search(json_pattern, response_text, re.DOTALL)
    
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    
    print(f"❌ Failed to parse JSON from response")
    return None


def extract_score_data(parsed_json: Dict[str, Any]) -> Dict[str, Any]:
    """
    Extract relevant fields for database storage
    """
    
    return {
        'overall_score': parsed_json.get('total_score', 0),
        'technical_skills_score': parsed_json.get('breakdown', {}).get('required_skills', 0) + 
                                   parsed_json.get('breakdown', {}).get('preferred_skills', 0),
        'experience_score': parsed_json.get('breakdown', {}).get('experience', 0),
        'education_score': parsed_json.get('breakdown', {}).get('education', 0),
        'project_relevance_score': parsed_json.get('breakdown', {}).get('additional', 0),
        'matching_skills': [item['skill'] for item in parsed_json.get('matched_skills_detail', [])],
        'missing_skills': parsed_json.get('missing_skills', []),
        'recommendations': parsed_json.get('recommendations', ''),
        'classification': parsed_json.get('classification', ''),
        'matched_required': parsed_json.get('matched_required', 0),
        'total_required': parsed_json.get('total_required', 0),
        'matched_preferred': parsed_json.get('matched_preferred', 0),
        'total_preferred': parsed_json.get('total_preferred', 0)
    }

In [ ]:
ats_Score_json=parse_ats_response(response)
extracted_data = extract_score_data(ats_Score_json)

In [19]:
extracted_data = extract_score_data(ats_Score_json)
extracted_data

{'overall_score': 83,
 'technical_skills_score': 48,
 'experience_score': 20,
 'education_score': 10,
 'project_relevance_score': 5,
 'matching_skills': ['Python',
  'Machine learning fundamentals',
  'scikit-learn',
  'PyTorch',
  'NLP concepts and transformer models',
  'APIs (FastAPI / Flask)',
  'Git and version control'],
 'missing_skills': [],
 'recommendations': "Consider adding explicit experience with SQL/relational databases, CI/CD pipelines (e.g., GitHub Actions, Jenkins), and MLOps tools like Airflow or Spark to strengthen the preferred skill set. Highlighting Docker usage in production deployments and any cloud architecture designs (e.g., AWS/GCP) would also align closely with the JD's expectations.",
 'classification': 'Excellent Match (80+)',
 'matched_required': 7,
 'total_required': 7,
 'matched_preferred': 2,
 'total_preferred': 5}

In [20]:
from autogen_agentchat.agents import AssistantAgent
from models.openrouter import getOpenRouterModel

SYSTEM_MESSAGE = """
You are an ATS Resume Improvement Agent.

You receive:
1. The ATS score breakdown produced by the ATS Resume Scoring Agent.
2. The Job Description analysis produced by the JD Analyst Agent.

Your task is to suggest specific, actionable resume improvements that can increase the ATS score.

CORE OBJECTIVE:
- Improve the resume’s ATS score by addressing gaps identified in the scoring breakdown.
- All suggestions must be directly tied to score deductions or missing requirements.

RULES:
- Do NOT invent or assume skills, experience, education, or certifications.
- Do NOT suggest adding information the candidate does not genuinely possess.
- Suggest improvements only if they are realistic resume edits (rewording, restructuring, clarifying existing experience, or adding missing but truthful details).
- Do NOT provide generic advice such as “add more keywords” without context.

IMPROVEMENT STRATEGY:
For each scoring category with lost points:
1. Identify the reason for score loss.
2. Explain briefly why this impacts ATS screening.
3. Suggest concrete resume changes the candidate can make.

SUGGESTION TYPES MAY INCLUDE:
- Rewording existing experience to better match JD terminology.
- Explicitly listing relevant tools or skills already implied but not stated.
- Moving critical skills or projects higher in the resume.
- Adding missing but genuine sections (e.g., Skills, Projects, Certifications).
- Clarifying project descriptions with technologies used.

PRIORITIZATION:
- Rank suggestions by expected ATS score impact (High / Medium / Low).
- Focus first on Required Skills and Experience gaps.

OUTPUT FORMAT:
Provide a structured response with:
- Category name (e.g., Required Skills, Experience, Education)
- Issue identified
- Suggested improvement
- Expected ATS impact (High / Medium / Low)

TONE:
- Professional, neutral, and practical.
- No motivational language.
- No assumptions about the candidate.

Your goal is to make the resume more ATS-aligned, not to rewrite the candidate’s career.

"""

def getImprovementAgent(model):
    improvement_agent = AssistantAgent(
        name="ImprovementAgent",
        model_client=model,
        description="An agent that suggests improvements to resumes based on ATS scoring results.",
        system_message=SYSTEM_MESSAGE)
    return improvement_agent

In [21]:
import json

task = f"""
ATS Score:{ats_Score_json}
"""
impro_agent = getImprovementAgent(model)
improvements = await impro_agent.run(task=task)

In [22]:
improvements.messages[1].content

'**Category:** Preferred Skills**Issue Identified:** Only 2 of the 5 preferred skills listed in the JD were clearly matched (e.g., “FastAPI” was found, but “CI/CD pipelines”, “MLOps tools”, “Docker”, and “cloud architecture” were not explicitly mentioned). This resulted in a lower score for the Preferred Skills category.  \n**Suggested Improvement:** Add a concise bullet under a “Key Projects” or “Technical Experience” section that explicitly states experience with any of the missing preferred tools/technologies. Examples of truthful wording:  \n- “Implemented continuous integration/continuous deployment (CI/CD) workflows using **GitHub Actions** for model training pipelines.”  \n- “Deployed models in production using **Docker** containers and managed scaling on **AWS** (EC2/EKS).”  \n- “Orchestrated batch training and monitoring with **Apache Airflow** (or **Spark**) as part of an MLOps pipeline.”  \nIf the candidate has only limited exposure, re‑phrase existing project descriptions t

In [30]:
import json
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="12345",
    database="resume_data"
)

cursor = conn.cursor()

def insert_resume_from_json(data):
    query = """
    INSERT INTO resumes (name, email, skills, experience, raw_json)
    VALUES (%s, %s, %s, %s, %s)
    """

    skills_str = json.dumps(data["skills"])  
    exp_str = json.dumps(data["experience"]) 

    values = (
        data["name"],
        data["email"],
        skills_str,
        exp_str,
        json.dumps(data)  
    )

    cursor.execute(query, values)
    conn.commit()
    return cursor.lastrowid 

def insert_job(jd):
    query = """
    INSERT INTO job_descriptions (title, required_skills, preferred_skills, jd_json)
    VALUES (%s, %s, %s, %s)
    """

    values = (
        json.dumps(jd["job_title"]),
        json.dumps(jd["required_skills"]),
        json.dumps(jd["preferred_skills"]),
        json.dumps(jd)
    )

    cursor.execute(query, values)
    conn.commit()
    return cursor.lastrowid 

def insert_score(score_json, resume_id, job_id):
    query = """
    INSERT INTO scores (resume_id, job_id, overall_score, breakdown)
    VALUES (%s, %s, %s, %s)
    """

    values = (
        resume_id,
        job_id,
        score_json["total_score"],
        json.dumps(score_json["breakdown"])
    )

    cursor.execute(query, values)
    conn.commit()

In [24]:
insert_resume_from_json(resume_details)

1

In [28]:
insert_job(jd_details)

1

In [ ]:
insert_score(ats_Score_json, resume_id=1, job_id=1)